In [ ]:
# Ruta del directorio actual
import os; ruta_actual = os.getcwd() 
# os.startfile(ruta_actual)

## Copiar concentración inicial "u_new0.out" a "u_new1.out" para MT3D

In [2]:
import shutil
shutil.copyfile('u_new0.out', 'u_new1.out')

'u_new1.out'

# (Ejecutar main_interfaz_local)(Construir BTN)(Ejecutar MT3D-USGS)

In [5]:
# Esta función genera un ciclo con los puntos 1) y 2).
# Ejecuta GWF-MT3D-NVG.ipynb haciendo 1): 
# 1) Lee conc_s.txt para modificar transport.btn y ejecutar GWF-MT3D-NVG.ipynb para generar conc_i.m3d.
# 2) Ejecuta RT.exe que lee conc_i.m3d, lo modifica y genera conc_s.txt
# 3) Se repite el ciclo [1),2)]
#******************************************************************************************************
# 1. Importación de librerías
import numpy as np
import subprocess
import time
import ipywidgets as widgets
from IPython.display import display

# 2. Rutas de archivos
#entrada_txt = "conc_s.txt"   # u_new.dat
#salida_m3d = "conc_i.m3d"    # u_tilde
entrada_txt = "u_new.out"     # u_new.out
salida_m3d = "u_tilde.dat"    # u_tilde.dat
main_interfaz = "main_interfaz_equi.exe" 
#ejecutable = "main_interfaz.exe" 

# 2.1
def sustituir_primera_linea(archivo_u_tilde="u_tilde.dat", archivo_conc="conc_i.m3d"):
    try:
        # Lee conc_i.m3d y colapsa TODOS los valores (en columna) a un solo renglón
        with open(archivo_conc, 'r') as f_conc:
            # Toma todos los tokens numéricos preservando su texto original
            tokens = []
            for line in f_conc:
                s = line.strip()
                if s:  # ignora líneas vacías
                    tokens.extend(s.split())  # separa por cualquier espacio/tab
        if not tokens:
            print(f"⚠️ {archivo_conc} no contiene valores.")
            return
        linea_conc = " ".join(tokens) + "\n"

        # Lee u_tilde.dat
        with open(archivo_u_tilde, 'r') as f_ut:
            lineas = f_ut.readlines()

        if len(lineas) < 2:
            print(f"⚠️ El archivo {archivo_u_tilde} no tiene al menos dos líneas.")
            return

        # Sustituye SOLO el primer renglón por los valores en una sola línea
        lineas[0] = linea_conc

        # Escribe de vuelta (dejando el 2° renglón intacto)
        with open(archivo_u_tilde, 'w') as f_ut:
            f_ut.writelines(lineas)

        print(f"✅ Sustitución completada: 1er renglón de '{archivo_u_tilde}' ← '{archivo_conc}' (columna→una línea)")

    except Exception as e:
        print(f"❌ Error sustituyendo primera línea: {e}")
        
#################################################################################
# 2.2
def sustituir_tercer_renglon_u_new(u_new="u_new.out", destino="u_new1.out"):
    """
    Toma el 3er renglón de u_new y lo sustituye como PRIMER renglón de destino (u_new1.out),
    preservando el resto de líneas de destino.
    """
    try:
        # Lee u_new.out
        with open(u_new, "r") as f_src:
            lineas_src = f_src.readlines()
        if len(lineas_src) < 3:
            print(f"⚠️ {u_new} no tiene al menos 3 líneas.")
            return
        tercera = lineas_src[2].rstrip("\n") + "\n"

        # Lee (o crea) u_new1.out
        try:
            with open(destino, "r") as f_dst:
                lineas_dst = f_dst.readlines()
        except FileNotFoundError:
            lineas_dst = []

        if len(lineas_dst) == 0:
            # si no existe o está vacío, crea con una línea y nada más
            lineas_dst = [tercera]
        else:
            # sustituye solo la primera línea
            lineas_dst[0] = tercera

        # Escribe de vuelta
        with open(destino, "w") as f_dst:
            f_dst.writelines(lineas_dst)

        print(f"✅ Sustitución completada: 1er renglón de '{destino}' ← 3er renglón de '{u_new}'")

    except Exception as e:
        print(f"❌ Error sustituyendo renglón: {e}")


##################################################################################
# 3. Slider para controlar nv
nv_slider = widgets.IntSlider(value=1, min=1, max=40, step=1, description="nv")
run_button = widgets.Button(description="Iniciar ciclo ")
output_area = widgets.Output()
contador_ejecuciones = 0

# 4. Función que ejecuta el ciclo
def ciclo_nv(b):
    global contador_ejecuciones
    output_area.clear_output()
    
    with output_area:
        contador_ejecuciones += 1
        print(f"🟢  Ejecución #{contador_ejecuciones}")        
        nv = nv_slider.value

        for i in range(nv):
            print(f"\n--- Iteración {i+1} de {nv} ---")
            
            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
            # EJECUTAR: (GWF,MT3D) ó GWF-MT3D-NVG
            # Condición según contador de ejecuciones
            if contador_ejecuciones == 1 and i == 0:
                print("▶️✅ Primera ejecución: corriendo GWF.ipynb y MT3D.ipynb...")
                %run GWF.ipynb  # Ejecutar GWF
                print(" 🟢🟢 Termina GWF, Inicia MT3D")
                %run MT3D.ipynb # Ejecutar MT3D
            else:
                print("▶️✅ Solo corriendo MT3D.ipynb...")
                %run MT3D.ipynb
                #%run GWF-MT3D-NVG.ipynb
# 1. La primera ejecución de MT3D se hace con "u_new1.out" que son las concentraciones en t=0
# 2. Se ejecuta MT3D-USGS, y se crea conc_i.m3d. 
# 3. Sustituir en u_tilde.dat la concentración (conc_i.m3d) dejando la segunda parte de u_tilde.dat sin cambio
            sustituir_primera_linea(archivo_u_tilde="u_tilde.dat", archivo_conc="conc_i.m3d")                
            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
                # Ejecutar (main_interfaz_equi.exe)
            try:
                print(f"✅ Ejecutando {main_interfaz} ...\n")
                proceso = subprocess.Popen([main_interfaz], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

                for linea in proceso.stdout:
                    print(linea, end="")  # imprime la salida línea por línea en tiempo real

                proceso.wait()
                if proceso.returncode != 0:
                    print(f"❌ El main_interfaz terminó con error. Código: {proceso.returncode}")
                    return  # o break, si estás dentro de un ciclo
            except Exception as e:
                print(f"❌ Error al ejecutar {main_interfaz}: {e}")
                return  # o break

# Despues de la ejecución de main_interfaz_equi.exe, se obtiene u_new.out 
# De u_new.out, se toma el tercer renglón y se crea u_new1.out para que las lea el 
# paquete BTN de MT3D y sea posible ejecutar nuevamente el MT3D.
            sustituir_tercer_renglon_u_new(u_new="u_new.out", destino="u_new1.out")
    
            print("✅ Construimos BTN, ejecutamos MT3D-USGS...\n")
            print("⏸️ Pausa de 1 segundo...\n")
            time.sleep(1)

# 5. Conectar botón
run_button.on_click(ciclo_nv)

# 6. Mostrar interfaz
display(nv_slider, run_button, output_area)

IntSlider(value=1, description='nv', max=40, min=1)

Button(description='Iniciar ciclo ', style=ButtonStyle())

Output()